# LSTM SIMULATION EXAMPLE
## This notebook demonstrates how to configure, initialize, and run LSTM model in a standalone mode. It includes environment setup, input preparation, and model execution
## Plots a comparison between simulated and observed streamflow, and calculates KGE and NSE

In [ ]:
# Ignore this section if you have already built and activated the `.bmi_lstm` environment.
# Build instructions (for VS Code)
# import sys, os
# from pathlib import Path
# lstm_dir = Path("~/Code/models/lstm").expanduser() 
#os.chdir( lstm_dir )
#!python -m pip install --upgrade pip
#%pip install -e .
#%pip install --no-cache-dir --force-reinstall netCDF4 # you may not need this
#!unalias python

In [ ]:
import sys, os
from pathlib import Path

print(sys.executable)  #--------------------------------------------------- check active Python environment

lstm_dir = Path("~/Code/models/lstm").expanduser() # --------------------- this should point to your local lstm directory 
os.chdir( lstm_dir )
print (os.getcwd())

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from netCDF4 import Dataset
from lstm import bmi_lstm  # Load module bmi_lstm (bmi_lstm.py) from lstm package.
import pickle
from hydrotools.metrics.metrics import *


In [ ]:
# Install, if needed
#!pip install matplotlib
#!pip install hydrotools
#pip install jupyter ipykernel
#python -m ipykernel install --user --name bmi_lstm --display-name "Python (bmi_lstm)"

In [ ]:
#bmi_cfg_file     = './configs/02064000_nh_NLDAS_hourly.yml'
bmi_cfg_file     = './configs/01022500_nh_NLDAS_hourly.yml'

sample_data_file = './data/usgs-streamflow-nldas_hourly.nc'

# creating an instance of an LSTM model
#print('Creating an instance of an BMI_LSTM model object')
model = bmi_lstm.bmi_LSTM()

# Initializing the BMI
#print('Initializing the BMI')
model.initialize(bmi_cfg_file)

# Get input data that matches the LSTM test runs
#print('Gathering input data')
sample_data = Dataset(sample_data_file, 'r')
sample_basins = {sample_data['basin'][x]:x for x in range(len(list(sample_data['basin'])))}
print (sample_basins)


print (model.cfg_bmi['basin_id'])
ibasin = sample_basins[model.cfg_bmi['basin_id']]

precip_data = sample_data['total_precipitation'][ibasin].data
temp_data   = sample_data['temperature'][ibasin].data
n_precip    = precip_data.size
runoff_output = np.zeros(n_precip)
print('Forcing data info:')
print('  n_precip =', n_precip)
print('  n_temp   =', temp_data.size)
print('  precip_data.dtype =', precip_data.dtype)
print('  temp_data.dtype   =', temp_data.dtype)
print('  precip:  min, max =', precip_data.min(), ',', precip_data.max() )
print('  temp:    min, max =', temp_data.min(), ',', temp_data.max() )


In [ ]:
for k in range(n_precip):
    precip = precip_data[k]
    temp   = temp_data[k]
    model.set_value('atmosphere_water__liquid_equivalent_precipitation_rate', precip)
    model.set_value('land_surface_air__temperature',temp+273.15)

    model.update()
    dest_array = np.zeros(1)
    #model.get_value('land_surface_water__runoff_volume_flux', dest_array)
    model.get_value('land_surface_water__runoff_depth', dest_array)
    runoff = dest_array[0]
    runoff_output[k] = runoff * 1000 #* 1000/3600.
    #print(' Temperature and precipitation are set to {:.2f} and {:.2f}'.format(temp, precip))
    #print(' Streamflow (cms) at time {} ({}) is {:.2f}'.format(model.get_current_time(), model.get_time_units(), runoff))

# Finalizing the BMI
model.finalize()

obs = sample_data['qobs_CAMELS_mm_per_hour'][ibasin].data

In [ ]:
start_plot=0
end_plot  =7000
plt.figure(figsize=(10, 5))
plt.plot(sample_data['qobs_CAMELS_mm_per_hour'][ibasin][start_plot:end_plot], label='Observed', c='k')
# plt.plot(runoff_output_list_limited[start_plot:end_plot],label='LSTM BMI')
plt.plot(runoff_output[start_plot:end_plot],label='LSTM BMI')

#plt.plot(nh_hourly_slope_mean_precip_temp_01022500[t2000+start_plot:t2000+end_plot],label='LSTM NH')
#plt.ylim([0,0.5])
plt.ylabel('streamflow mm/hr')
plt.xlabel('hours')
plt.legend()
plt.show()
plt.close()


In [ ]:
kge = kling_gupta_efficiency(obs, runoff_output)
nse = nash_sutcliffe_efficiency(obs, runoff_output)
print ("KGE: ", kge)
print ("NSE: ", nse)